# E08 — RAG Failure Analysis — Diagnostic (no LLM calls, A2 not modified)

**Research question**: What failure modes remain in A2 Standard RAG, and which of them could
plausibly be addressed by selective agentic investigation?

E08 is diagnostic only — it does not improve the system. `retrieval_v1` and
`classification_prompt_v1` are read-only inputs here, never modified or re-invoked for
classification. The only re-queries performed are local, deterministic, non-LLM (BM25 +
cross-encoder rerank, both already used throughout E06/E07), used purely to diagnose the 6
reranker-limited cases.

**Terminology, used consistently throughout**: `source_valid_evidence` (the existing validator's
finding — verbatim, present in context, structurally valid) is kept strictly separate from
`gold_evidence_overlap` (does the quote actually overlap the true annotated gold span). The
former never implies the latter.

In [1]:
import csv
import json
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().resolve().parents[1] if Path.cwd().name == "E08_rag_failure_analysis" else Path.cwd()
sys.path.insert(0, str(REPO_ROOT))
E08 = REPO_ROOT / "experiments/E08_rag_failure_analysis"
E07 = REPO_ROOT / "experiments/E07_standard_rag"
RESULTS = E08 / "results"

quant = json.load(open(RESULTS / "agent_opportunity_quantification.json"))
rows = list(csv.DictReader(open(RESULTS / "rag_failure_analysis.csv")))
print("Manual-review union size:", quant["manual_review_union_size"])
print("Reviewed error count:", quant["reviewed_error_count"])

Manual-review union size: 73
Reviewed error count: 63


## 2. Frozen E07 inputs — no reruns

In [2]:
e07_summary = json.load(open(E07 / "results/run_E07_A2_train.json"))
print("E07 accuracy:", e07_summary["classification"]["accuracy"])
print("E07 total errors (150 - correct):", 150 - int(round(e07_summary["classification"]["accuracy"]*150)))
print("Retrieval-aware counts from E07:", e07_summary["retrieval_aware_failure_counts"])

E07 accuracy: 0.43333333333333335
E07 total errors (150 - correct): 85
Retrieval-aware counts from E07: {'B_reasoning_limited': 79, 'n/a (correct)': 51, 'C_evidence_selection_failure': 14, 'A_retrieval_limited': 6}


## 3. Terminology refinement: source-valid vs. gold-overlapping evidence

The 54 cases E07 called "wrong-label-but-valid-evidence" only confirmed the quotes were
*source-valid* (verbatim, structurally correct) -- not that they were the *correct* evidence.

In [3]:
print("wrong-label + source-valid evidence: 54 total")
print("  Group A (gold_evidence_overlap=True):", quant.get("wrong_valid_group_a_gold_overlap", "see summary.md"))
print("  Group B (gold_evidence_overlap=False):", quant.get("wrong_valid_group_b_no_overlap", "see summary.md"))
a_count = sum(1 for r in rows if r['gold_label'] in ('Entailment','Contradiction')
              and r['predicted_label'] != r['gold_label'] and r['source_valid_evidence']=='True'
              and r['gold_evidence_overlap']=='True')
b_count = sum(1 for r in rows if r['gold_label'] in ('Entailment','Contradiction')
              and r['predicted_label'] != r['gold_label'] and r['source_valid_evidence']=='True'
              and r['gold_evidence_overlap']=='False')
print("recomputed from CSV -- Group A:", a_count, " Group B:", b_count)

wrong-label + source-valid evidence: 54 total
  Group A (gold_evidence_overlap=True): 4
  Group B (gold_evidence_overlap=False): 50
recomputed from CSV -- Group A: 4  Group B: 50


## 4-5. Overall and Contradiction-specific errors

In [4]:
print("Overall reviewed errors:", quant["reviewed_error_count"])
print("Quantification overall:", quant["quantification_overall"])
print("Quantification overall %:", quant["quantification_overall_pct"])
print()
print("Quantification Contradiction:", quant["quantification_contradiction"])
print("Quantification Contradiction %:", quant["quantification_contradiction_pct"])

Overall reviewed errors: 63
Quantification overall: {'model_reasoning_limited': 31, 'agentically_fixable': 26, 'static_pipeline_fixable': 6}
Quantification overall %: {'model_reasoning_limited': 49.2, 'agentically_fixable': 41.3, 'static_pipeline_fixable': 9.5}

Quantification Contradiction: {'model_reasoning_limited': 19, 'agentically_fixable': 9, 'static_pipeline_fixable': 1}
Quantification Contradiction %: {'model_reasoning_limited': 65.5, 'agentically_fixable': 31.0, 'static_pipeline_fixable': 3.4}


## 6. Reranker/filtering-limited cases (renamed from "retrieval-limited")

All 6 cases had gold evidence within BM25's own top-20 candidate pool -- the lexical stage did
not fail. The cross-encoder reranker demoted the correct candidate below the frozen top-5
cutoff in every case.

In [5]:
for cid, d in quant["reranker_diag"].items():
    print(f"{cid}: BM25 rank={d['bm25_top20_rank']}  post-rerank rank={d['post_rerank_rank_of_20']}  "
          f"top-10 would include={d['would_top10_include']}")
print()
print("All 6 classified STATIC_PIPELINE_FIXABLE -- a fixed top-k change (not an agent) would recover them.")
print("retrieval_v1 was NOT changed -- this is a candidate intervention, not implemented.")

train::160::nda-10: BM25 rank=2  post-rerank rank=11  top-10 would include=False
train::247::nda-10: BM25 rank=6  post-rerank rank=9  top-10 would include=True
train::353::nda-10: BM25 rank=9  post-rerank rank=9  top-10 would include=True
train::379::nda-10: BM25 rank=3  post-rerank rank=8  top-10 would include=True
train::438::nda-2: BM25 rank=6  post-rerank rank=8  top-10 would include=True
train::518::nda-10: BM25 rank=1  post-rerank rank=9  top-10 would include=True

All 6 classified STATIC_PIPELINE_FIXABLE -- a fixed top-k change (not an agent) would recover them.
retrieval_v1 was NOT changed -- this is a candidate intervention, not implemented.


## 7. All 54 wrong-label + source-valid-evidence cases

In [6]:
wrong_valid_rows = [r for r in rows if r['gold_label'] in ('Entailment','Contradiction')
                     and r['predicted_label'] != r['gold_label'] and r['source_valid_evidence']=='True']
print(f"n={len(wrong_valid_rows)}")
from collections import Counter
print("failure_primary breakdown within this subgroup:", Counter(r['failure_primary'] for r in wrong_valid_rows))

n=54
failure_primary breakdown within this subgroup: Counter({'MODEL_REASONING_LIMITED': 26, 'AGENTICALLY_FIXABLE': 23, 'STATIC_PIPELINE_FIXABLE': 5})


## 8. Manual-review methodology

Deterministic union (seed=1000, fixed before any content was inspected): ALL 6 reranker-limited,
ALL 29 Contradiction failures, ALL 54 wrong-label+source-valid-evidence cases, plus a 10-case
sample of E05→E07 gains and an 8-case sample of E05→E07 regressions not already covered by the
mandatory categories. **Final unique count: 73 (48.7% of the full 150-case set).**

For each reviewed failure, the 10 brief-mandated questions were answered structurally via the
computed fields: gold-context presence, source-vs-gold evidence distinction, sufficiency,
missing-context identification, static-vs-dynamic fixability, and (critically) whether a
plausible action **changes the available information** rather than merely re-asking the same
model to reconsider the same five chunks -- "reason again" alone was never accepted as agent
justification.

## 9-10. Failure taxonomy and static/agentic/model-limited resolution

Category D (EVIDENCE_SELECTION_LIMITED) was never left as its own bucket -- every such case was
resolved into STATIC_PIPELINE_FIXABLE, AGENTICALLY_FIXABLE, or MODEL_REASONING_LIMITED based on
whether the gold-relevant chunk *itself* (not the whole 5-chunk context) contained a genuine
disambiguating cue.

In [7]:
print(Counter(r['failure_primary'] for r in rows if r['correct']!='True'))

Counter({'MODEL_REASONING_LIMITED': 31, 'AGENTICALLY_FIXABLE': 26, 'STATIC_PIPELINE_FIXABLE': 6})


## 11. Runtime-observable signals

Checked on the whole 5-chunk context (the realistically-implementable version, since a real
system does not know which chunk is gold).

In [8]:
print(quant["runtime_signal_frequencies"])
print()
print("Exception/carve-out language appears in 62/63 reviewed errors -- far too frequent to")
print("serve as a standalone escalation trigger without additional conditioning.")

{'cross_reference_cue,exception_carveout_cue': 26, 'cross_reference_cue,exception_carveout_cue,defined_term_missing_cue': 5, 'exception_carveout_cue': 16, 'exception_carveout_cue,defined_term_missing_cue': 12, 'cross_reference_cue,exception_carveout_cue,low_score_margin': 3, 'low_score_margin,low_score_margin': 1}

Exception/carve-out language appears in 62/63 reviewed errors -- far too frequent to
serve as a standalone escalation trigger without additional conditioning.


## 12. E05→E07 gains/regressions — diagnostic reading (no reruns)

In [9]:
e05_cases = {json.loads(l)["case_id"]: json.loads(l) for l in open(E07.parent / "E05_full_context/results/run_E05_A1_train_cases.jsonl")}
e07_cases = {json.loads(l)["case_id"]: json.loads(l) for l in open(E07 / "results/run_E07_A2_train_cases.jsonl")}
for cid in quant["e05_regress_sample"][:3]:
    e05c, e07c = e05_cases[cid], e07_cases[cid]
    print(f"{cid}  gold={e05c['gold_label']}  E05_pred={e05c['predicted_label']}  E07_pred={e07c['predicted_label']}")

train::187::nda-10  gold=NotMentioned  E05_pred=NotMentioned  E07_pred=Contradiction
train::222::nda-18  gold=NotMentioned  E05_pred=NotMentioned  E07_pred=Entailment
train::264::nda-12  gold=NotMentioned  E05_pred=NotMentioned  E07_pred=Contradiction


## 13. Oracle-action analysis (evaluator-only diagnostic labels)

In [10]:
print(Counter(r['oracle_action_needed'] for r in rows if r['correct']!='True'))

Counter({'none_identified': 31, 'search_exception': 20, 'expand_candidate_window': 6, 'follow_cross_reference': 3, 'retrieve_definition': 3})


## 14. Representative cases

In [11]:
for fam in ("STATIC_PIPELINE_FIXABLE", "AGENTICALLY_FIXABLE", "MODEL_REASONING_LIMITED"):
    print(f"--- {fam} ---")
    for r in [r for r in rows if r['failure_primary']==fam][:2]:
        print(f"  {r['case_id']}  {r['gold_label']}->{r['predicted_label']}  oracle={r['oracle_action_needed']}")

--- STATIC_PIPELINE_FIXABLE ---
  train::160::nda-10  Entailment->NotMentioned  oracle=expand_candidate_window
  train::247::nda-10  Entailment->NotMentioned  oracle=expand_candidate_window
--- AGENTICALLY_FIXABLE ---
  train::141::nda-7  Contradiction->NotMentioned  oracle=search_exception
  train::145::nda-12  Entailment->Contradiction  oracle=follow_cross_reference
--- MODEL_REASONING_LIMITED ---
  train::102::nda-1  Contradiction->NotMentioned  oracle=none_identified
  train::102::nda-19  Contradiction->NotMentioned  oracle=none_identified


## 15. Quantified opportunity buckets (headline numbers for E09)

| | Overall | Contradiction |
|---|---|---|
| STATIC_PIPELINE_FIXABLE | 9.5% | 3.4% |
| AGENTICALLY_FIXABLE | 41.3% | 31.0% |
| MODEL_REASONING_LIMITED | 49.2% | 65.5% |
| Evidence/output/ambiguous other | 0% | 0% |

## 16. E08 conclusion: **B — NARROW SELECTIVE AGENT POTENTIALLY JUSTIFIED**

Not A: 41.3% of errors show a genuine, evaluator-confirmed, cue-linked information gap — too
large to dismiss. Not C: the largest bucket (49.2% overall, 65.5% Contradiction) is
model-reasoning-limited with no identified fix, and the runtime-observable trigger candidate
(exception/carve-out presence, 62/63 errors) is too undiscriminating for broad escalation.
**A narrow agent, tightly conditioned beyond raw cue presence, targeting the reranker-limited
and genuinely-cued evidence-selection subsets, is the evidence-supported recommendation** — not
a final A3 decision, the evidence base for E09.

## 17. Recommended next diagnostic (not executed here)

A matched stronger-model (GPT-5 mini) diagnostic on the frozen A2 setup, before committing to
substantial A3 implementation, to test whether the large MODEL_REASONING_LIMITED bucket
resolves with a stronger model alone — potentially a simpler, cheaper intervention than a
selective agent. **Not run in E08**; requires separate explicit authorization (would be
reconstruction-v2's first hosted-model call in the A1/A2 architecture line).